# 02 · Feature Engineering and SageMaker Feature Store

Two feature groups, keyed by `app_id`:

| Feature group | Contents | Used by |
|---|---|---|
| `steam-draft-features` | cleaned description, length, price, age rating, platforms, developer/publisher, release date, 22-tag label | Tag QA Gate, Reception |
| `steam-post-release-features` | positive/negative counts, peak CCU, playtime, owners, rating | analysis only; **never** the tag model |

The separation is the leakage guard. The training snapshot for the pipeline is queried from the **offline store**
(Athena) and written to `curated/tag_gate_snapshot/`.

In [ ]:
# Pin the SageMaker SDK v2 API used by this project (no-op if already installed)
%pip install -q "sagemaker>=2.232,<3"

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
def find_repo_root():
    """The notebooks import from src/, so the whole repo must be cloned, not just notebooks/."""
    candidates = [os.path.expanduser("~/steam-store-copilot")]
    try:  # the kernel's working directory may have been deleted (e.g. after re-cloning)
        candidates = [os.getcwd(), os.path.dirname(os.getcwd())] + candidates
    except FileNotFoundError:
        os.chdir(os.path.expanduser("~"))
    for path in candidates:
        if os.path.isdir(os.path.join(path, "src")) and os.path.exists(os.path.join(path, "src", "config.py")):
            return path
    raise RuntimeError("Could not find the repo's src/ folder. Clone the full repository in Studio "
                       "(Git > Clone Repository) and open the notebooks from steam-store-copilot/notebooks/.")


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

import boto3
import sagemaker

from src.config import S3Layout

# Works both inside SageMaker Studio and on a laptop with Learner Lab credentials in ~/.aws/credentials
boto_session = boto3.Session(region_name=boto3.Session().region_name or "us-east-1")
sess = sagemaker.Session(boto_session=boto_session)
region = sess.boto_region_name
try:
    role = sagemaker.get_execution_role(sagemaker_session=sess)
except Exception:  # running outside SageMaker: use the Learner Lab execution role
    account = boto_session.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/LabRole"
layout = S3Layout(bucket=sess.default_bucket())
sm = boto3.client("sagemaker", region_name=region)
print("Region:", region)
print("Role:  ", role)
print("Bucket:", layout.bucket)

In [ ]:
import time

import pandas as pd

from src.config import DRAFT_FEATURE_GROUP, POST_RELEASE_FEATURE_GROUP
from src.data.loaders import load_recs_games, load_store_games
from src.data.preprocess import build_tag_dataset
from src.data.s3io import fetch, write_csv
from src.features import feature_store as fs

games = load_store_games(fetch(f"{layout.raw}/store/games.csv"))
recs_games = load_recs_games(fetch(f"{layout.raw}/recs/games.csv"))
tag_ds = build_tag_dataset(games)

event_time = time.time()
draft_df = fs.draft_feature_frame(tag_ds, event_time)
post_df = fs.post_release_feature_frame(games, recs_games, tag_ds["app_id"], event_time)
print("draft:", draft_df.shape, "| post-release:", post_df.shape)
draft_df.head(3)

In [ ]:
pd.DataFrame({"column": draft_df.columns, "dtype": draft_df.dtypes.astype(str).values})

## Create the feature groups
Online store enabled (low-latency lookup at draft time) plus an offline store in S3 for training snapshots.

In [ ]:
draft_fg = fs.create_feature_group(
    DRAFT_FEATURE_GROUP, draft_df, layout.offline_store, role, sess,
    description="Draft-time features + 22-tag labels (no post-release signal)")
post_fg = fs.create_feature_group(
    POST_RELEASE_FEATURE_GROUP, post_df, layout.offline_store, role, sess,
    description="Post-release signals - excluded from the tag model to prevent leakage")

In [ ]:
fs.ingest(draft_fg, draft_df, max_workers=8)
fs.ingest(post_fg, post_df, max_workers=8)

## Online store lookup
The same game in both groups: the draft group has only what a studio knows before launch.

In [ ]:
runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)
app_id = str(draft_df["app_id"].iloc[0])
for name in (DRAFT_FEATURE_GROUP, POST_RELEASE_FEATURE_GROUP):
    rec = runtime.get_record(FeatureGroupName=name, RecordIdentifierValueAsString=app_id)["Record"]
    print(f"--- {name}")
    for f in rec:
        print(f"  {f['FeatureName']:<24} {f['ValueAsString'][:80]}")

In [ ]:
desc = draft_fg.describe()
print("Offline store:", desc["OfflineStoreConfig"]["S3StorageConfig"]["ResolvedOutputS3Uri"])
print("Glue table:   ", desc["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"])
pd.DataFrame(desc["FeatureDefinitions"])

## Training snapshot from the offline store (Athena)
The offline store usually becomes queryable 5-15 minutes after ingestion. The cell waits up to 20 minutes and
deduplicates to the latest record per game.

In [ ]:
deadline = time.time() + 20 * 60
while not fs.offline_store_ready(draft_fg, min_rows=int(0.99 * len(draft_df))):
    if time.time() > deadline:
        raise TimeoutError("Offline store not ready yet - re-run this cell in a few minutes")
    print("waiting for offline store ...")
    time.sleep(60)

query = draft_fg.athena_query()
query.run(query_string=fs.snapshot_query(query.table_name),
          output_location=f"s3://{layout.bucket}/athena-results/")
query.wait()
snapshot = query.as_dataframe()
print(f"Snapshot rows from offline store: {len(snapshot):,}")
snapshot.head(3)

In [ ]:
snapshot_uri = write_csv(snapshot, f"{layout.snapshot}/snapshot.csv", index=False)
print("Training snapshot written to", snapshot_uri)
assert set(fs.DRAFT_COLUMNS) <= set(snapshot.columns)
leaky = {"positive", "negative", "peak_ccu", "avg_playtime_forever", "estimated_owners"}
assert not leaky & set(snapshot.columns), "post-release features leaked into the snapshot!"
print("Leakage check passed: no post-release columns in the tag-gate snapshot.")